# Nested CV con splits Cx explícitos — cuatro escenarios de generalización

Pipeline de evaluación de regresión logística + embeddings AlphaFold siguiendo los **cuatro escenarios de Park & Marcotte (2012, *Nature Methods*)** definidos en el esquema de selección de negativos:

| Escenario | Descripción | Generalización evaluada |
|-----------|-------------|------------------------|
| **C1** | Efector y proteína ya vistos en train | Más optimista (baseline) |
| **C2E** | Efector nuevo, proteína vista | Generalización sobre efectores |
| **C2P** | Proteína nueva, efector visto | Generalización sobre proteínas |
| **C3** | Ambos nuevos | Más exigente — uso real en inferencia |

Los splits de C2E, C2P y C3 se cargan directamente desde `generate_cx_splits.py` (evitando recomputar y garantizando reproducibilidad). C1 usa `StratifiedKFold` estándar como baseline sin restricción de leakage.

Al final del notebook se entrena un **modelo final** con todos los datos etiquetados y se realiza **inferencia sobre los casos dudosos**, indicando el nivel de confianza de cada predicción (C1/C2E/C2P/C3) según si el modelo ha visto moléculas similares durante el entrenamiento.

In [1]:
import sys
import json
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (
    GridSearchCV, cross_validate,
    BaseCrossValidator, StratifiedKFold,
)

# Generador de splits Cx (debe estar en el mismo directorio o en PYTHONPATH)
sys.path.insert(0, str(Path(".").resolve()))
from generate_cx_splits import build_and_save_splits, load_splits

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR      = Path("output_Efectores_alphafold_all")
OUTPUT_RESULTS = Path("results_NestedCV_Cx/results_NestedCV_nivel_estricto_g13sep")
# PATH_LABELED    = OUTPUT_DIR / "labeled_interactions"    # pares etiquetados (pos + neg)
# PATH_UNKNOWN    = OUTPUT_DIR / "unknown_interactions"    # pares dudosos para inferencia
PATH_METADATA   = OUTPUT_RESULTS / "metadata.csv"           # sample_name, effector_group, protein_group, label
SPLITS_DIR      = OUTPUT_RESULTS / "splits"                 # directorio donde se guardan/leen los splits Cx

# Tipo de embedding y pooling por defecto (se sobreescribe tras la búsqueda)
DEFAULT_EMB_TYPE = "single_embeddings"

Preparamos los Data Frames para el entrenamiento y la inferencia que se usarán a continuación.

In [3]:
df_total = pd.read_csv("Interacciones_EffectorProteina_LiteratureOnly_Ordenadas_NleG.csv")
# Añadimos los grupos de efectores y proteínas
effector_groups = pd.read_csv("effector_groups_function.csv")
mapping_ef_groups = effector_groups.set_index("Effector")["effector_group"]
df_total["Effector_Group"] = df_total["Effector"].map(mapping_ef_groups)
protein_groups = pd.read_csv("protein_go_80_clusters_grupo13sep.csv")
mapping_prot_groups = protein_groups.set_index("gene_name")["protein_group"]
df_total["Protein_Group"] = df_total["ProteinGeneName"].map(mapping_prot_groups)
# Añadimos además una columna Sample name con el prefijo de las carpetas generadas por AlphaFold3
df_total["Sample_name"] = df_total["Protein"] + "_" + df_total["Effector"]
df_total.head()

,Effector,Protein,ProteinGeneName,Shared_Connections,Shortest_Path,Is_Connected,Effector_Group,Protein_Group,Sample_name
0,EspL,O89110,Casp8,4,1.0,True,Cluster 3,GO_cluster_13.3,O89110_EspL
1,EspL,Q60855,Ripk1,3,1.0,True,Cluster 3,GO_cluster_13.3,Q60855_EspL
2,NleB,O89110,Casp8,2,1.0,True,Cluster 1,GO_cluster_13.3,O89110_NleB
3,NleA,Q8R4B8,Nlrp3,1,1.0,True,Cluster 6,GO_cluster_14,Q8R4B8_NleA
4,NleA,Q9D8T2,Gsdmd,1,1.0,True,Cluster 6,GO_cluster_13.1,Q9D8T2_NleA


In [4]:
# Modificamos el Data Frame para que tenga solo la información que nos interesa
df_total = df_total.drop(columns=['Protein', 'Shared_Connections', 'Shortest_Path'])
df_total.rename(columns={'ProteinGeneName': 'Protein'}, inplace=True)
df_total.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Sample_name
0,EspL,Casp8,True,Cluster 3,GO_cluster_13.3,O89110_EspL
1,EspL,Ripk1,True,Cluster 3,GO_cluster_13.3,Q60855_EspL
2,NleB,Casp8,True,Cluster 1,GO_cluster_13.3,O89110_NleB
3,NleA,Nlrp3,True,Cluster 6,GO_cluster_14,Q8R4B8_NleA
4,NleA,Gsdmd,True,Cluster 6,GO_cluster_13.1,Q9D8T2_NleA


#### Pulido de las carpetas de output

En el output de AlphaFold3 hay carpetas que no fue capaz de ejecutar, con lo que se quedaron a medias. Esas carpetas nos interesa quitarlas del entrenamiento y de la inferencia porque no podemos trabajar con esos datos.

Localizamos qué muestras son y las quitamos del análisis.

In [5]:
incomplete_samples = set()

for sample in OUTPUT_DIR.iterdir():
    if sample.is_dir() and ".ipynb_checkpoints" not in sample.name:
        if not (sample / "TERMS_OF_USE.md") in sample.iterdir():
            # Nos quedamos con el nombre limpio y lo guardamos en la lista
            parts = sample.name.split('_')
            clean_name = "_".join(parts[:2])
            incomplete_samples.add(clean_name)

len(incomplete_samples)
incomplete_samples

{'B2RWS6_EspN',
 'B2RWS6_NleL',
 'P26039_EspL',
 'P26039_EspN',
 'P26039_NleL',
 'P26039_Tir'}

In [6]:
# Quitamos todas esas parejas que no ha conseguido procesar de la lista de interacciones totales y nos olvidamos de ellas
df_total = df_total[~df_total["Sample_name"].isin(incomplete_samples)]
len(df_total)

5466

In [7]:
# Creamos un Data Frame para las interacciones que formarán parte del train
# Añadimos una columna de Sample name que coincidirá con el prefijo de la carpeta generada por AlphaFold3
df_labeled = pd.read_csv("Interacciones_Entrenamiento_CV_estricto_g13sep.csv")
uniprot_equivalences = pd.read_csv("Interacciones_EffectorProteina_LiteratureOnly_Ordenadas_NleG.csv", sep=",")
prot_id = pd.Series(uniprot_equivalences.Protein.values, index=uniprot_equivalences.ProteinGeneName).to_dict()
df_labeled["Protein_ID"] = df_labeled["Protein"].map(prot_id)
df_labeled["Sample_name"] = df_labeled["Protein_ID"] + "_" + df_labeled["Effector"]
# y nos quedamos con las mismas columnas que en el Data Frame total
df_labeled = df_labeled.drop(columns=['Pathways_Shared', 'Shared_Connections', 'Shortest_Path', 'Interaction_Score', 'Protein_ID'])
df_labeled.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Label,Sample_name
0,NleB,Ripk1,True,Cluster 1,GO_cluster_13.3,1,Q60855_NleB
1,NleB,Nfkb1,True,Cluster 1,GO_cluster_14,1,P25799_NleB
2,NleD,Mapkapk2,True,Cluster 2,GO_cluster_05,1,P49138_NleD
3,NleB,Traf6,True,Cluster 1,GO_cluster_14,1,P70196_NleB
4,NleB,Traf5,True,Cluster 1,GO_cluster_12,1,P70191_NleB


In [8]:
# Repetimos ahora con las interacciones desconocidas que se usarán en la inferencia
train_samples = set(df_labeled["Sample_name"])
df_unknown = df_total[~df_total["Sample_name"].isin(train_samples)]
df_unknown.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Sample_name
0,EspL,Casp8,True,Cluster 3,GO_cluster_13.3,O89110_EspL
1,EspL,Ripk1,True,Cluster 3,GO_cluster_13.3,Q60855_EspL
3,NleA,Nlrp3,True,Cluster 6,GO_cluster_14,Q8R4B8_NleA
25,NleD,Mapk14,True,Cluster 2,GO_cluster_07,P47811_NleD
27,NleD,Mapk1,True,Cluster 2,GO_cluster_07,P63085_NleD


In [9]:
# Comprobamos que las longitudes son correctas
print(f"Parejas etiquetadas: {len(df_labeled)}")
print(f"Parejas cuya interacción se desconoce: {len(df_unknown)}")
print(f"Parejas totales: {len(df_total)}")
print(f"Suma de parejas etiquetadas y desconocidas: {len(df_labeled)} + {len(df_unknown)} = {len(df_labeled) + len(df_unknown)}")

Parejas etiquetadas: 197
Parejas cuya interacción se desconoce: 5269
Parejas totales: 5466
Suma de parejas etiquetadas y desconocidas: 197 + 5269 = 5466


In [10]:
# Por último añadimos una columna de etiquetas a las parejas que se van a usar en el entrenamiento para saber si son positivas o negativas
df_labeled["Label"] = df_labeled["Is_Connected"].astype(int)
df_unknown["Label"] = "-"
df_total["Label"] = df_total.apply(
    lambda x: int(x["Is_Connected"]) if x["Sample_name"] in train_samples else "-",
    axis=1
)

df_total.to_csv("Interacciones_totales_CV.csv")
df_labeled.to_csv("Interacciones_entrenamiento_relajado_CV.csv")
df_unknown.to_csv("Interacciones_inferencia_relajado_CV.csv")

df_labeled.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Label,Sample_name
0,NleB,Ripk1,True,Cluster 1,GO_cluster_13.3,1,Q60855_NleB
1,NleB,Nfkb1,True,Cluster 1,GO_cluster_14,1,P25799_NleB
2,NleD,Mapkapk2,True,Cluster 2,GO_cluster_05,1,P49138_NleD
3,NleB,Traf6,True,Cluster 1,GO_cluster_14,1,P70196_NleB
4,NleB,Traf5,True,Cluster 1,GO_cluster_12,1,P70191_NleB


# Funciones

### 1. Carga de embeddings

Funciones para cargar embeddings de AlphaFold en *chunks* y construir bloques
(X, y, sample_names) para el entrenamiento. La función `load_block` también devuelve
los `sample_names` necesarios para mapear a los splits Cx.

In [11]:
# def load_samples_in_chunks(input_dir, batch_size=2, emb_type="single_embeddings", transforms=None):
#     """
#     Generator que carga embeddings y etiquetas en batches para eficiencia de memoria.

#     :param input_dir:   Directorio raíz con las carpetas de cada muestra.
#     :param batch_size:  Número de muestras por batch.
#     :param emb_type:    Clave del array dentro del .npz (p.ej. 'single_embeddings').
#     :param transforms:  Lista opcional de funciones a aplicar al embedding cargado.
#     :yields: Tuple (np.ndarray de embeddings del batch, lista de etiquetas).
#     """
#     def load_sample(name):
#         path = input_dir / name / "seed-0_embeddings" / (name + "_seed-0_embeddings.npz")
#         embeddings = np.load(path)
#         label = int(name.split("_")[-1])
#         return embeddings[emb_type], label

#     sample_names = sorted([d.name for d in input_dir.iterdir() if d.is_dir()])

#     for i in range(0, len(sample_names), batch_size):
#         batch = sample_names[i:i + batch_size]
#         out, labels = [], []
#         for name in batch:
#             repr_emb, label = load_sample(name)
#             if transforms is None:
#                 out.append(repr_emb)
#             elif len(transforms) == 1:
#                 out.append(transforms[0](repr_emb))
#             else:
#                 out.append([t(repr_emb) for t in transforms])
#             labels.append(label)
#         yield np.asarray(out), labels


# def load_block(input_dir, emb_type="single_embeddings", transforms=None):
#     """
#     Carga todos los embeddings y etiquetas de un directorio completo.
#     Devuelve también sample_names (necesarios para alinear con splits Cx).

#     :returns: X (np.ndarray), y (np.ndarray), sample_names (list[str])
#     """
#     X, y = [], []
#     sample_names = sorted([d.name for d in input_dir.iterdir() if d.is_dir()])

#     for sample, labels in load_samples_in_chunks(
#         input_dir, emb_type=emb_type, transforms=transforms
#     ):
#         X.append(sample)
#         y.extend(labels)

#     return np.concatenate(X, axis=0), np.array(y), sample_names

In [12]:
# def load_samples_in_chunks_no_label(input_dir, batch_size=2, emb_type="single_embeddings", transforms=None):
#     """Generator igual a load_samples_in_chunks pero sin etiquetas (para dudosos)."""
#     def load_sample(name):
#         path = input_dir / name / "seed-0_embeddings" / (name + "_seed-0_embeddings.npz")
#         return np.load(path)[emb_type]

#     sample_names = sorted([d.name for d in input_dir.iterdir() if d.is_dir()])
#     for i in range(0, len(sample_names), batch_size):
#         batch = sample_names[i:i + batch_size]
#         out = []
#         for name in batch:
#             repr_emb = load_sample(name)
#             if transforms is None:
#                 out.append(repr_emb)
#             elif len(transforms) == 1:
#                 out.append(transforms[0](repr_emb))
#             else:
#                 out.append([t(repr_emb) for t in transforms])
#         yield np.asarray(out)


# def load_block_no_label(input_dir, emb_type="single_embeddings", transforms=None):
#     """Carga embeddings sin etiquetas (casos dudosos para inferencia).

#     :returns: X (np.ndarray), sample_names (list[str])
#     """
#     X = []
#     sample_names = sorted([d.name for d in input_dir.iterdir() if d.is_dir()])
#     for sample in load_samples_in_chunks_no_label(
#         input_dir, emb_type=emb_type, transforms=transforms
#     ):
#         X.append(sample)
#     return np.concatenate(X, axis=0), sample_names

Funciones para cargar embeddings de AlphaFold de acuerdo a las muestras presentes en el Data Frame proporcionado.

In [13]:
def load_block_from_csv(input_dir, target_df, emb_type="single_embeddings", transforms=None):
    import numpy as np
    X, y, sample_names = [], [], []

    for _, row in target_df.iterrows():
        sample_id = str(row['Sample_name']).strip()

        # --- CAMBIO AQUÍ: Buscar carpetas que EMPIECEN por el ID ---
        # Esto encontrará "ID", "ID_0", "ID_1", etc.
        matching_folders = list(input_dir.glob(f"{sample_id}*"))
        # Filtrar para asegurarnos de que sean directorios
        matching_folders = [f for f in matching_folders if f.is_dir()]

        if matching_folders:
            # Tomamos la primera coincidencia encontrada
            folder_path = matching_folders[0]

            # Buscamos el .npz dentro
            npz_folder = folder_path / "seed-0_embeddings"
            npz_files = list(npz_folder.glob("*.npz")) if npz_folder.exists() else []

            if npz_files:
                try:
                    path = npz_files[0]
                    emb_data = np.load(path)[emb_type]

                    if transforms is None:
                        X.append(emb_data)
                    elif len(transforms) == 1:
                        X.append(transforms[0](emb_data))
                    else:
                        X.append([t(emb_data) for t in transforms])

                    sample_names.append(sample_id)
                    if 'Label' in target_df.columns:
                        y.append(int(row['Label']))

                except Exception as e:
                    print(f"❌ Error al cargar datos de {sample_id}: {e}")
        else:
            # print(f"❓ No se encontró carpeta para: {sample_id}")
            pass

    X_array = np.asarray(X)
    if X_array.ndim > 2 and transforms and len(transforms) > 1:
        X_array = np.swapaxes(X_array, 0, 1)

    print(f"✅ Éxito: Se cargaron {len(X)} muestras de las {len(target_df)} del Excel.")
    return X_array, (np.array(y) if y else None), sample_names

### 2. Splits Cx — generación y carga

`build_and_save_splits` (de `generate_cx_splits.py`) genera y serializa los splits
C3/C2E/C2P a disco. `load_splits` los recupera.

`PrecomputedSplitter` traduce los splits (listas de `sample_name`) a índices de array,
haciéndolos compatibles con `cross_validate` y `GridSearchCV` de scikit-learn.

In [14]:
class PrecomputedSplitter(BaseCrossValidator):
    """
    Splitter de scikit-learn que usa splits precalculados (de generate_cx_splits).

    Traduce listas de sample_name a índices de numpy para compatibilidad
    con GridSearchCV y el bucle externo manual de hpm_search_nested_cv.
    """

    def __init__(self, folds: dict, sample_names: list):
        self.folds = folds
        self.name2idx = {name: i for i, name in enumerate(sample_names)}

    def _resolve(self, names):
        """Convierte lista de sample_name a array de índices (ignorando ausentes)."""
        return np.array(
            [self.name2idx[n] for n in names if n in self.name2idx],
            dtype=int,
        )

    def _iter_test_indices(self, X=None, y=None, groups=None):
        for fold in self.folds.values():
            yield self._resolve(fold["test"])

    def split(self, X, y=None, groups=None):
        for fold in self.folds.values():
            train_idx = self._resolve(fold["train"])
            test_idx  = self._resolve(fold["test"])
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return len(self.folds)

    def _iter_test_masks(self, X=None, y=None, groups=None):
        n = X.shape[0] if X is not None else len(self.name2idx)
        for test_idx in self._iter_test_indices(X, y, groups):
            mask = np.zeros(n, dtype=bool)
            if len(test_idx):
                mask[test_idx] = True
            yield mask


def get_outer_splitter(scenario: str, sample_names: list, splits_dir) -> BaseCrossValidator:
    """
    Devuelve el splitter externo apropiado para cada escenario:
      C1  → StratifiedKFold(5) — sin restricción de leakage (baseline).
      C2E → leave-one-effector_group-out  (precomputado).
      C2P → leave-one-protein_group-out   (precomputado).
      C3  → leave-one-(eg × pg)-out       (precomputado).
    """
    if scenario == "C1":
        return StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    folds = load_splits(str(splits_dir), scenario)
    return PrecomputedSplitter(folds, sample_names)


def get_inner_groups(groups_eg: np.ndarray, groups_pg: np.ndarray, scenario: str) -> np.ndarray:
    """
    Construye el array de grupos para el bucle INTERNO (GridSearchCV) a partir
    de los grupos del subconjunto de train del fold externo.

    El inner loop debe respetar la misma restricción que el outer para no
    introducir leakage en la selección de hiperparámetros
    (Krstajic et al., 2014, J Cheminform; Varoquaux et al., 2017, NeuroImage).

    Mapeo por escenario:
      C1  → None           (StratifiedKFold, sin grupos).
      C2E → groups_eg      (leave-one-effector_group-out en el inner train).
      C2P → groups_pg      (leave-one-protein_group-out en el inner train).
      C3  → eg + '__' + pg (leave-one-(eg × pg)-out en el inner train).
    """
    if scenario == "C1":
        return None
    if scenario == "C2E":
        return groups_eg
    if scenario == "C2P":
        return groups_pg
    if scenario == "C3":
        return np.array([f"{eg}__{pg}" for eg, pg in zip(groups_eg, groups_pg)])
    raise ValueError(f"Escenario desconocido: {scenario}")


def get_inner_splitter(scenario: str, inner_groups: np.ndarray, inner_cv: int = 5):
    """
    Devuelve (splitter_inner, groups_para_fit) para el GridSearchCV interno.

    Para C2E/C2P/C3 se usa LeaveOneGroupOut con los grupos del subconjunto de
    train, garantizando que la búsqueda de hiperparámetros no ve combinaciones
    no vistas en el fold de train externo.

    Para C1 se usa StratifiedKFold sin grupos.
    """
    from sklearn.model_selection import LeaveOneGroupOut
    if scenario == "C1":
        return StratifiedKFold(n_splits=inner_cv, shuffle=True, random_state=42), None
    return LeaveOneGroupOut(), inner_groups


### 3. Verificación de splits

Antes de entrenar, se imprime el resumen de cada fold: tamaño de train/test,
clases presentes y (para C3) muestras excluidas. Los folds con una sola clase
producirán NaN en las métricas durante el CV; se contabilizan y se informa.

In [15]:
def verify_cx_splits(scenario: str, splitter, X, y, sample_names, splits_dir):
    """
    Comprueba que PrecomputedSplitter ha cargado exactamente las particiones
    generadas por generate_cx_splits, comparando contra los metadatos JSON.

    Solo imprime el resumen final y los folds que no cuadren (si los hay).
    El reporte completo fold a fold ya está en splits_report.txt.

    :returns: (n_valid, n_one_class, n_empty)
    """
    import json as _json
    from pathlib import Path as _Path

    # C1 no tiene metadatos Cx — solo contar folds válidos
    if scenario == "C1":
        n_valid = n_one_class = n_empty = 0
        for train_idx, test_idx in splitter.split(X, y):
            if len(test_idx) == 0:
                n_empty += 1
            elif len(set(y[test_idx])) < 2:
                n_one_class += 1
            else:
                n_valid += 1
        viab = "✅ viable" if n_valid >= 3 else "❌ inviable"
        print(f"  [{scenario}] {splitter.get_n_splits(X, y)} folds  "
              f"válidos={n_valid}  una clase={n_one_class}  vacíos={n_empty}  {viab}")
        return n_valid, n_one_class, n_empty

    # Cargar metadatos (fuente de verdad de generate_cx_splits)
    meta_path = _Path(splits_dir) / f"splits_{scenario}_meta.json"
    with open(meta_path) as f:
        meta = _json.load(f)

    fold_labels = list(splitter.folds.keys())
    n_valid = n_one_class = n_empty = mismatches = 0
    mismatch_lines = []

    for label, (train_idx, test_idx) in zip(fold_labels, splitter.split(X, y)):
        if len(test_idx) == 0:
            n_empty += 1
            continue

        pos = int(y[test_idx].sum())
        neg = int(len(test_idx) - pos)

        if pos == 0 or neg == 0:
            n_one_class += 1
        else:
            n_valid += 1

        m = meta.get(str(label), {})
        errors = []
        if m.get("n_test_pos") != pos:
            errors.append(f"pos: got {pos} expected {m.get('n_test_pos')}")
        if m.get("n_test_neg") != neg:
            errors.append(f"neg: got {neg} expected {m.get('n_test_neg')}")
        if m.get("n_train") != len(train_idx):
            errors.append(f"train: got {len(train_idx)} expected {m.get('n_train')}")
        if errors:
            mismatches += 1
            mismatch_lines.append(f"    ❌ fold [{label}]: {', '.join(errors)}")

    viab = "✅ viable" if n_valid >= 3 else "❌ inviable"
    cross = "✅ coincide con report_splits" if mismatches == 0             else f"❌ {mismatches} fold(s) NO coinciden con report_splits"
    print(f"  [{scenario}] {len(fold_labels)} folds  "
          f"válidos={n_valid}  una clase={n_one_class}  vacíos={n_empty}  "
          f"{viab}  |  {cross}")
    for line in mismatch_lines:
        print(line)

    return n_valid, n_one_class, n_empty


### 4. Nested Cross-Validation multi-escenario

`hpm_search_nested_cv` es la función central: bucle externo con el splitter del
escenario elegido y bucle interno con `StratifiedKFold` para la búsqueda de
hiperparámetros. Los folds vacíos o de una clase retornan NaN (gestionados por
`np.nanmean` en el análisis posterior).

`test_pooling_transforms_all_scenarios` itera sobre todos los tipos de embedding,
poolings y escenarios en un único paso, devolviendo un diccionario
`results[scenario][emb_name]`.

In [16]:
def hpm_search_nested_cv(
    X, y,
    outer_splitter,
    groups_eg: np.ndarray,
    groups_pg: np.ndarray,
    scenario: str,
    inner_cv: int = 5,
):
    """
    Nested CV con bucle EXTERNO manual y bucle INTERNO consistente con el escenario.

    El bucle externo se implementa manualmente (sin cross_validate) para poder
    construir en cada fold los groups internos restringidos al subconjunto de train.
    Esto es necesario porque cross_validate no soporta groups dinámicos por fold.

    Bucle externo : outer_splitter (PrecomputedSplitter o StratifiedKFold).
    Bucle interno : LeaveOneGroupOut con grupos del subconjunto de train
                    para C2E / C2P / C3; StratifiedKFold para C1.

    Importante: usar el mismo tipo de split en ambos bucles es el requisito
    estándar para nested CV con datos agrupados (Krstajic et al., 2014,
    J Cheminform; Varoquaux et al., 2017, NeuroImage).

    :param X:             Array de embeddings (n_samples, n_features).
    :param y:             Array de etiquetas.
    :param outer_splitter: BaseCrossValidator para el bucle externo.
    :param groups_eg:     Array de grupos de efector (len = n_samples).
    :param groups_pg:     Array de grupos de proteína (len = n_samples).
    :param scenario:      'C1', 'C2E', 'C2P' o 'C3'.
    :param inner_cv:      n_splits para StratifiedKFold en C1.
    :returns: Dict con claves 'test_accuracy', 'test_roc_auc', 'test_pr_auc',
              'estimator' (lista de GridSearchCV ajustados por fold).
    """
    from sklearn.model_selection import LeaveOneGroupOut, GridSearchCV
    from sklearn.metrics import (balanced_accuracy_score, roc_auc_score, average_precision_score, matthews_corrcoef, precision_recall_curve, f1_score, precision_score, recall_score)
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    
    param_grid = [
        {
            "logisticregression__solver":  ["lbfgs", "newton-cg"],
            "logisticregression__penalty": ["l2"],
            "logisticregression__C":       [0.001, 0.01, 0.1, 1, 10],
        },
        {
            "logisticregression__solver":  ["liblinear"],
            "logisticregression__penalty": ["l1"],
            "logisticregression__C":       [0.001, 0.01, 0.1, 1, 10],
        },
    ]

    pipeline = make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=10000, tol=1e-4, class_weight='balanced'),
    )

    metrics_keys = (
        "test_bal_accuracy_50", "test_mcc_50", "test_precision_50", "test_recall_50", "test_pr_gain_50", "test_f1g_50",
        "test_bal_accuracy_opt", "test_mcc_opt", "test_precision_opt", "test_recall_opt", "test_pr_gain_opt", "test_f1g_opt",
        "test_best_threshold", "test_roc_auc", "test_pr_auc", "test_lift", "test_auprg", "test_expected_f1g"
    )
    
    results = {k: [] for k in metrics_keys}
    results["estimator"] = []
    results["fold_details"] = []

    # Extraer fold_ids si el splitter es PrecomputedSplitter (Cx); usar índice para C1
    if hasattr(outer_splitter, "folds"):
        fold_ids = list(outer_splitter.folds.keys())
    else:
        fold_ids = [f"fold_{i}" for i in range(outer_splitter.get_n_splits(X, y))]

    for fold_id, (train_idx, test_idx) in zip(fold_ids, outer_splitter.split(X, y)):
        # ── Fold vacío o de una sola clase → NaN ─────────────────────────────
        if len(test_idx) == 0:
            for k in ("test_bal_accuracy", "test_mcc", "test_roc_auc", "test_pr_auc"):
                results[k].append(np.nan)
                results["estimator"].append(None)
                results["fold_details"].append({"fold_id": fold_id, "n_test": 0,
                "n_test_pos": 0, "n_test_neg": 0,
                "bal_accuracy": np.nan, "mcc": np.nan, "roc_auc": np.nan, "pr_auc": np.nan,
                "best_C": np.nan, "best_penalty": "", "note": "vacío"})
            continue

        y_test = y[test_idx]
        
        if len(np.unique(y_test)) < 2:
            for k in metrics_keys: results[k].append(np.nan)
            results["estimator"].append(None)
            results["fold_details"].append({"fold_id": fold_id, "note": "una clase"})
            continue

        X_train, y_train = X[train_idx], y[train_idx]
        X_test            = X[test_idx]

        # ── Construir grupos del subconjunto de train para el inner CV ────────
        eg_train = groups_eg[train_idx]
        pg_train = groups_pg[train_idx]
        inner_groups = get_inner_groups(eg_train, pg_train, scenario)
        inner_splitter, fit_groups = get_inner_splitter(scenario, inner_groups, inner_cv)

        # ── Inner CV: búsqueda de hiperparámetros ────────────────────────────
        grid = GridSearchCV(
            pipeline, param_grid,
            cv=inner_splitter,
            scoring="average_precision",   # PR AUC como criterio de selección
            n_jobs=-1,
            error_score=np.nan,
        )
        grid.fit(X_train, y_train, groups=fit_groups)

        # ── Evaluación en test externo ────────────────────────────────────────
        try:
            proba = grid.predict_proba(X_test)[:, 1]
            pi = float(y_test.sum() / len(y_test)) # Prevalencia de positivos

            # 1. EVALUACIÓN CON UMBRAL FIJO (0.50)
            pred_50 = (proba >= 0.5).astype(int)
            bal_acc_50 = balanced_accuracy_score(y_test, pred_50)
            mcc_50     = matthews_corrcoef(y_test, pred_50)
            prec_50    = precision_score(y_test, pred_50, zero_division=0)
            rec_50     = recall_score(y_test, pred_50, zero_division=0)
            f1_50      = f1_score(y_test, pred_50, zero_division=0)
            
            pr_gain_50 = (prec_50 - pi) / ((1.0 - pi) * prec_50) if prec_50 > pi else 0.0
            f1g_50     = (f1_50 - pi) / ((1.0 - pi) * f1_50) if f1_50 > pi else 0.0

            # 2. BARRIDO DE UMBRALES PARA ENCONTRAR EL MEJOR (MAX F1)
            thresholds = np.linspace(0.01, 0.99, 100)
            best_f1 = -1.0
            best_thresh = 0.5
            
            for t in thresholds:
                current_pred = (proba >= t).astype(int)
                current_f1 = f1_score(y_test, current_pred, zero_division=0)
                if current_f1 > best_f1:
                    best_f1 = current_f1
                    best_thresh = t

            # 3. EVALUACIÓN CON UMBRAL ÓPTIMO
            pred_opt = (proba >= best_thresh).astype(int)
            bal_acc_opt = balanced_accuracy_score(y_test, pred_opt)
            mcc_opt     = matthews_corrcoef(y_test, pred_opt)
            prec_opt    = precision_score(y_test, pred_opt, zero_division=0)
            rec_opt     = recall_score(y_test, pred_opt, zero_division=0)
            f1_opt      = best_f1

            pr_gain_opt = (prec_opt - pi) / ((1.0 - pi) * prec_opt) if prec_opt > pi else 0.0
            f1g_opt     = (f1_opt - pi) / ((1.0 - pi) * f1_opt) if f1_opt > pi else 0.0

            # 4. CURVA COMPLETA E INTEGRACIÓN GEOMÉTRICA (AUPRG y E[FG1])
            roc     = roc_auc_score(y_test, proba)
            pr      = average_precision_score(y_test, proba)
            lift    = pr / pi if pi > 0 else np.nan

            precisions, recalls, _ = precision_recall_curve(y_test, proba)
            with np.errstate(divide='ignore', invalid='ignore'):
                prec_g_curve = (precisions - pi) / ((1.0 - pi) * precisions)
                rec_g_curve = (recalls - pi) / ((1.0 - pi) * recalls)

            prec_g_curve = np.clip(np.nan_to_num(prec_g_curve, nan=0.0), 0.0, 1.0)
            rec_g_curve = np.clip(np.nan_to_num(rec_g_curve, nan=0.0), 0.0, 1.0)

            sort_idx = np.argsort(rec_g_curve)
            rec_g_sorted = rec_g_curve[sort_idx]
            prec_g_sorted = prec_g_curve[sort_idx]
            
            auprg = float(np.trapz(prec_g_sorted, rec_g_sorted))

            # Extracción de y0 (Precision Gain cuando Recall Gain == 0)
            zero_rec_indices = np.where(rec_g_sorted == 0.0)[0]
            y0 = float(prec_g_sorted[zero_rec_indices[-1]]) if len(zero_rec_indices) > 0 else 0.0

            # Cálculo del F1-Gain Esperado 
            if y0 == 1.0:
                expected_f1g = auprg / 2.0 + 0.25
            else:
                num = auprg / 2.0 + 0.25 - (pi * (1.0 - y0**2)) / 4.0
                den = 1.0 - pi * (1.0 - y0)
                expected_f1g = float(num / den) if den != 0 else 0.0

            # Guardar en estructura general
            results["test_bal_accuracy_50"].append(bal_acc_50)
            results["test_mcc_50"].append(mcc_50)
            results["test_precision_50"].append(prec_50)
            results["test_recall_50"].append(rec_50)
            results["test_pr_gain_50"].append(pr_gain_50)
            results["test_f1g_50"].append(f1g_50)
            
            results["test_bal_accuracy_opt"].append(bal_acc_opt)
            results["test_mcc_opt"].append(mcc_opt)
            results["test_precision_opt"].append(prec_opt)
            results["test_recall_opt"].append(rec_opt)
            results["test_pr_gain_opt"].append(pr_gain_opt)
            results["test_f1g_opt"].append(f1g_opt)
            
            results["test_best_threshold"].append(best_thresh)
            results["test_roc_auc"].append(roc)
            results["test_pr_auc"].append(pr)
            results["test_lift"].append(lift)
            results["test_auprg"].append(auprg)
            results["test_expected_f1g"].append(expected_f1g)

            # Extraer hiperparámetros del mejor estimador
            bp = grid.best_params_ if hasattr(grid, "best_params_") else {}
            c_key   = next((k for k in bp if k.endswith("__C")), None)
            pen_key = next((k for k in bp if k.endswith("__penalty")), None)
            
            results["fold_details"].append({
                "fold_id":      fold_id,
                "n_test":       len(test_idx),
                "n_test_pos":   int(y_test.sum()),
                "n_test_neg":   int(len(test_idx) - y_test.sum()),
                "roc_auc":      round(roc, 4),
                "pr_auc":       round(pr, 4),
                "lift":         round(lift, 4) if not np.isnan(lift) else np.nan,
                "auprg":        round(auprg, 4),
                "expected_f1g": round(expected_f1g, 4),
                "best_threshold": round(best_thresh, 4),
                
                "bal_accuracy_50": round(bal_acc_50, 4),
                "mcc_50":          round(mcc_50, 4),
                "precision_50":    round(prec_50, 4),
                "recall_50":       round(rec_50, 4),
                "pr_gain_50":      round(pr_gain_50, 4),
                "f1g_50":          round(f1g_50, 4),
                
                "bal_accuracy_opt": round(bal_acc_opt, 4),
                "mcc_opt":          round(mcc_opt, 4),
                "precision_opt":    round(prec_opt, 4),
                "recall_opt":       round(rec_opt, 4),
                "pr_gain_opt":      round(pr_gain_opt, 4),
                "f1g_opt":          round(f1g_opt, 4),
                
                "best_C":       bp.get(c_key, np.nan) if c_key else np.nan,
                "best_penalty": bp.get(pen_key, "")   if pen_key else "",
                "note":         "ok",
            })
            
        except Exception:
            for k in metrics_keys: results[k].append(np.nan)
            results["fold_details"].append({"fold_id": fold_id, "note": f"error: {str(e)}"})
            
        results["estimator"].append(grid)

    # ── Resumen de folds ──────────────────────────────────────────────────────
    acc_arr  = np.array(results["test_bal_accuracy_opt"], dtype=float)
    n_total  = len(acc_arr)
    n_valid  = int((~np.isnan(acc_arr)).sum())
    print(f"    folds: total={n_total}  válidos={n_valid}  NaN={n_total - n_valid}")

    return results


In [17]:
def test_pooling_transforms_all_scenarios(transforms, labeled_dir, splits_dir,
                                           metadata_df: pd.DataFrame,
                                           scenarios=None):
    """
    Evalúa todas las combinaciones (escenario × embedding × pooling) con Nested CV.

    El bucle interno de cada escenario usa la misma restricción de grupo que
    el bucle externo (C2E→efector, C2P→proteína, C3→celda, C1→estratificado).

    :param transforms:   Dict {emb_type: [lista de funciones de pooling]}.
    :param labeled_dir:  Path al directorio de muestras etiquetadas.
    :param splits_dir:   Path donde se guardaron los splits Cx.
    :param metadata_df:  DataFrame con columnas sample_name, effector_group, protein_group.
    :param scenarios:    Lista de escenarios a evaluar (por defecto los 4).
    :returns: Dict anidado all_results[scenario][emb_name] con métricas por pooling.
    """

    import time
    from collections import defaultdict
                                               
    if scenarios is None:
        scenarios = ["C1", "C2E", "C2P", "C3"]

    all_results = {}

    # Preconstruir mapas de grupo a partir de metadata_df
    eg_map = dict(zip(metadata_df["Sample_name"], metadata_df["Effector_Group"]))
    pg_map = dict(zip(metadata_df["Sample_name"], metadata_df["Protein_Group"]))

    for scenario in scenarios:
        print(f"\n{'#'*70}")
        print(f"#  ESCENARIO {scenario}")
        print(f"{'#'*70}")
        all_results[scenario] = {}

        for emb_name, transform_list in transforms.items():
            print(f"\n{'='*65}")
            print(f"  Embedding: {emb_name}  ({len(transform_list)} poolings)")
            print(f"{'='*65}")

            X_full, y_full, sample_names = load_block_from_csv(
                labeled_dir, target_df=metadata_df[metadata_df['Label'].isin([0,1])], emb_type=emb_name, transforms=transform_list
            )
            print(X_full, y_full)

            # Arrays de grupos por muestra — alineados con X_full
            groups_eg = np.array([eg_map.get(n, "UNKNOWN") for n in sample_names])
            groups_pg = np.array([pg_map.get(n, "UNKNOWN") for n in sample_names])

            # Splitter externo
            outer_splitter = get_outer_splitter(scenario, sample_names, splits_dir)

            # SI EL SHAPE ES (3, 394, 384), necesitamos mover el 394 al principio
            if X_full.ndim == 3 and X_full.shape[0] != len(y_full):
                # Esto cambia (3, 394, 384) -> (394, 3, 384)
                X_full = np.transpose(X_full, (1, 0, 2))

            # Verificar splits (consume el iterador; reconstruir después)
            verify_cx_splits(scenario, outer_splitter, X_full, y_full, sample_names, splits_dir=splits_dir)
            outer_splitter = get_outer_splitter(scenario, sample_names, splits_dir)

            res_emb = defaultdict(list)

            for i, _ in enumerate(transform_list):
                pooling_name = ["mean", "max", "min"][i] if i < 3 else f"pooling_{i}"
                print(f"\n  --- {emb_name} | {pooling_name} ---")

                Xi = X_full[:, i, :] if X_full.ndim == 3 else X_full

                # Reconstruir splitter externo para cada pooling
                outer_splitter = get_outer_splitter(scenario, sample_names, splits_dir)

                t0 = time.time()
                nested_res = hpm_search_nested_cv(
                    Xi, y_full,
                    outer_splitter=outer_splitter,
                    groups_eg=groups_eg,
                    groups_pg=groups_pg,
                    scenario=scenario,
                )
                elapsed = time.time() - t0

                # Extracción para estadísticas limpias
                bal50   = np.array(nested_res["test_bal_accuracy_50"], dtype=float)
                mcc50   = np.array(nested_res["test_mcc_50"],           dtype=float)
                pr50   = np.array(nested_res["test_precision_50"],     dtype=float)
                roc50   = np.array(nested_res["test_recall_50"],        dtype=float)
                prg50  = np.array(nested_res["test_pr_gain_50"],       dtype=float)
                f50   = np.array(nested_res["test_f1g_50"],           dtype=float)
                
                balopt  = np.array(nested_res["test_bal_accuracy_opt"], dtype=float)
                mccopt  = np.array(nested_res["test_mcc_opt"],          dtype=float)
                propt  = np.array(nested_res["test_precision_opt"],    dtype=float)
                rocopt  = np.array(nested_res["test_recall_opt"],       dtype=float)
                prgopt = np.array(nested_res["test_pr_gain_opt"],      dtype=float)
                fopt  = np.array(nested_res["test_f1g_opt"],          dtype=float)
                th    = np.array(nested_res["test_best_threshold"],   dtype=float)
                
                roc   = np.array(nested_res["test_roc_auc"],         dtype=float)
                pr    = np.array(nested_res["test_pr_auc"],          dtype=float)
                lift  = np.array(nested_res["test_lift"],            dtype=float)
                auprg = np.array(nested_res["test_auprg"],           dtype=float)
                f1g_e = np.array(nested_res["test_expected_f1g"],    dtype=float)

                n_valid_folds = int((~np.isnan(bal50)).sum())

                # Almacenar promedios
                res_emb["pooling_name"].append(pooling_name)
                res_emb["cv_roc_auc"].append(np.nanmean(roc))
                res_emb["cv_pr_auc"].append(np.nanmean(pr))
                res_emb["cv_lift"].append(np.nanmean(lift))
                res_emb["cv_auprg"].append(np.nanmean(auprg))
                res_emb["cv_expected_f1g"].append(np.nanmean(f1g_e))
                
                res_emb["cv_bal_accuracy_50"].append(np.nanmean(bal50))
                res_emb["cv_mcc_50"].append(np.nanmean(mcc50))
                res_emb["cv_precision_50"].append(np.nanmean(pr50))
                res_emb["cv_recall_50"].append(np.nanmean(roc50))
                res_emb["cv_pr_gain_50"].append(np.nanmean(prg50))
                res_emb["cv_f1g_50"].append(np.nanmean(f50))
                
                res_emb["cv_bal_accuracy_opt"].append(np.nanmean(balopt))
                res_emb["cv_mcc_opt"].append(np.nanmean(mccopt))
                res_emb["cv_precision_opt"].append(np.nanmean(propt))
                res_emb["cv_recall_opt"].append(np.nanmean(rocopt))
                res_emb["cv_pr_gain_opt"].append(np.nanmean(prgopt))
                res_emb["cv_f1g_opt"].append(np.nanmean(fopt))
                res_emb["cv_best_threshold"].append(np.nanmean(th))
                
                res_emb["n_valid_folds"].append(n_valid_folds)
                res_emb["opt_time"].append(elapsed)
                res_emb["estimators"].append(nested_res["estimator"])
                res_emb["fold_details"].append(nested_res["fold_details"])

                # --- COMPARATIVA DETALLADA IMPRESA EN TIEMPO REAL ---
                print(f"    Folds válidos: {n_valid_folds} | Tiempo: {elapsed:.1f}s")
                print(f"    ROC AUC: {np.nanmean(roc):.4f} | PR AUC: {np.nanmean(pr):.4f} | Lift: {np.nanmean(lift):.2f}x")
                print(f"    AUPRG (Área PR-Gain): {np.nanmean(auprg):.4f} | E[FG1] (F1-Gain Esperado): {np.nanmean(f1g_e):.4f}")
                print(f"    [PUNTOS OPERATIVOS CONCRETOS]")
                print(f"      -> Umbral Fijo [0.50]:")
                print(f"         Prec: {np.nanmean(pr50):.4f} | Rec: {np.nanmean(roc50):.4f} | Bal.Acc: {np.nanmean(bal50):.4f} | PR-Gain: {np.nanmean(prg50):.4f} | F1-Gain: {np.nanmean(f50):.4f}")
                print(f"      -> Umbral Optimizado [Promedio: {np.nanmean(th):.3f}]:")
                print(f"         Prec: {np.nanmean(propt):.4f} | Rec: {np.nanmean(rocopt):.4f} | Bal.Acc: {np.nanmean(balopt):.4f} | PR-Gain: {np.nanmean(prgopt):.4f} | F1-Gain: {np.nanmean(fopt):.4f}")
                print(f"    " + "-"*58)
                
            all_results[scenario][emb_name] = res_emb

    return all_results


### 5. Prueba de aleatoriedad

Verifica que el pipeline no aprende de artefactos estructurales: se asignan
etiquetas aleatorias y se espera accuracy ≈ 0.50. Se usa el escenario C1
(StratifiedKFold) para rapidez.

In [18]:
def sanity_check_random_labels(X, y, n_repeats=3):
    """
    Entrena con etiquetas aleatorias para confirmar ausencia de leakage estructural.
    Usa StratifiedKFold (C1) como escenario de comprobación rápida.

    Si el accuracy > 0.55, hay leakage en la representación o en el pipeline.
    """
    outer_c1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    print("\n=== Sanity Check: Labels Aleatorias (escenario C1) ===")
    accs = []
    for i in range(n_repeats):
        y_rand = np.random.permutation(y)
        outer_c1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=i)
        res = hpm_search_nested_cv(X, y_rand, outer_c1)
        acc = np.nanmean(res["test_bal_accuracy"])
        accs.append(acc)
        print(f"  Repetición {i+1}: balanced accuracy = {acc:.4f}")
    print(f"  Media: {np.mean(accs):.4f}  (esperado ≈ 0.50)")
    print("=" * 45)

### 6. Tabla de resultados por escenario

In [19]:
def optresults2table_nested(all_results):
    """
    Convierte el dict de resultados (todos los escenarios) en un DataFrame.

    Cada fila corresponde a una combinación (escenario × embedding × pooling).
    Se extrae el hiperparámetro más frecuente entre los folds externos válidos.

    :param all_results: Dict devuelto por test_pooling_transforms_all_scenarios.
    :returns: pd.DataFrame ordenado por (escenario, pr_auc desc).
    """
    import pandas as pd
    
    rows = []
    for scenario, scenario_res in all_results.items():
        for emb_name, res in scenario_res.items():
            for i in range(len(res["pooling_name"])):

                estimators = [
                    e for e in res["estimators"][i]
                    if hasattr(e, "best_params_")
                ]

                if estimators:
                    # Prefijo dinámico: funciona para LogisticRegression
                    # dentro de pipelines con o sin warm start
                    param_keys = list(estimators[0].best_params_.keys())
                    c_key  = next((k for k in param_keys if k.endswith("__C")), None)
                    pen_k  = next((k for k in param_keys if k.endswith("__penalty")), None)
                    sol_k  = next((k for k in param_keys if k.endswith("__solver")), None)

                    cs        = [e.best_params_[c_k]  for e in estimators for c_k  in [c_key]  if c_k]
                    penalties = [e.best_params_[p_k]  for e in estimators for p_k  in [pen_k]  if p_k]
                    solvers   = [e.best_params_[s_k]  for e in estimators for s_k  in [sol_k]  if s_k]
                else:
                    cs = penalties = solvers = []

                rows.append({
                    "scenario":       scenario,
                    "representation": emb_name,
                    "pooling":        res["pooling_name"][i],
                    "roc_auc":          res["cv_roc_auc"][i],
                    "pr_auc":           res["cv_pr_auc"][i],
                    "pr_auc_lift":      res["cv_lift"][i],
                    "auprg":            res["cv_auprg"][i],
                    "expected_f1_gain": res["cv_expected_f1g"][i],
                    
                    # Umbral 50
                    "precision_50":     res["cv_precision_50"][i],
                    "recall_50":        res["cv_recall_50"][i],
                    "bal_accuracy_50":  res["cv_bal_accuracy_50"][i],
                    "mcc_50":           res["cv_mcc_50"][i],
                    "pr_gain_50":       res["cv_pr_gain_50"][i],
                    "f1_gain_50":       res["cv_f1g_50"][i],
                    
                    # Umbral Optimizado
                    "best_threshold":   res["cv_best_threshold"][i],
                    "precision_opt":    res["cv_precision_opt"][i],
                    "recall_opt":       res["cv_recall_opt"][i],
                    "bal_accuracy_opt": res["cv_bal_accuracy_opt"][i],
                    "mcc_opt":          res["cv_mcc_opt"][i],
                    "pr_gain_opt":      res["cv_pr_gain_opt"][i],
                    "f1_gain_opt":      res["cv_f1g_opt"][i],
                    
                    "time_s":           res["opt_time"][i],
                    "C_most_freq":    max(set(cs), key=cs.count) if cs else np.nan,
                    "penalty":        max(set(penalties), key=penalties.count) if penalties else "",
                    "solver":         max(set(solvers), key=solvers.count) if solvers else "",
                    "n_valid_folds":  res["n_valid_folds"][i],
                })

    df = pd.DataFrame(rows)
    return df.sort_values(["scenario", "pr_auc"], ascending=[True, False]).reset_index(drop=True)

### 6b. Tabla de resultados por fold individual

Muestra qué grupo o combinación estaba en el test set de cada fold, su composición (n_pos, n_neg) y las métricas obtenidas. Especialmente útil cuando hay pocos folds (C3 con 4 folds) y la varianza entre folds es informativa por sí misma.

In [20]:
def fold_detail_table(all_results: dict) -> pd.DataFrame:
    """
    Genera una tabla con los resultados detallados por fold individual.

    Para cada combinación (escenario × embedding × pooling × fold) muestra:
      - fold_id  : grupo o combinación dejada en test (e.g. "EG1__PG3" en C3)
      - n_test   : número total de muestras en el test set de ese fold
      - n_test_pos / n_test_neg : composición del test set
      - accuracy, roc_auc, pr_auc : métricas de ese fold concreto
      - best_C, best_penalty : hiperparámetros seleccionados en el inner CV
      - note     : 'ok' | 'una clase' | 'vacío' | 'error'

    Útil especialmente en C3 con pocos folds: permite ver qué combinación
    biológica produce mejor o peor rendimiento y por qué (tamaño, desbalance).

    :param all_results: Dict devuelto por test_pooling_transforms_all_scenarios.
    :returns: pd.DataFrame ordenado por (scenario, representation, pooling, fold_id).
    """
    import pandas as pd
    
    rows = []
    for scenario, scenario_res in all_results.items():
        for emb_name, res in scenario_res.items():
            for i, pooling_name in enumerate(res["pooling_name"]):
                for fold_info in res["fold_details"][i]:
                    rows.append({
                        "scenario":       scenario,
                        "representation": emb_name,
                        "pooling":        pooling_name,
                        "fold_id":        fold_info["fold_id"],
                        "note":           fold_info["note"],
                        "n_test":         fold_info.get("n_test", 0),
                        "n_test_pos":     fold_info.get("n_test_pos", 0),
                        "roc_auc":        fold_info.get("roc_auc", np.nan),
                        "pr_auc":         fold_info.get("pr_auc", np.nan),
                        "lift":           fold_info.get("lift", np.nan),
                        "auprg":          fold_info.get("auprg", np.nan),
                        "expected_f1g":   fold_info.get("expected_f1g", np.nan),
                        
                        # Detalles 50
                        "precision_50":    fold_info.get("precision_50", np.nan),
                        "recall_50":       fold_info.get("recall_50", np.nan),
                        "bal_accuracy_50": fold_info.get("bal_accuracy_50", np.nan),
                        "mcc_50":          fold_info.get("mcc_50", np.nan),
                        "pr_gain_50":      fold_info.get("pr_gain_50", np.nan),
                        "f1g_50":          fold_info.get("f1g_50", np.nan),
                        
                        # Detalles Opt
                        "best_threshold":  fold_info.get("best_threshold", np.nan),
                        "precision_opt":   fold_info.get("precision_opt", np.nan),
                        "recall_opt":      fold_info.get("recall_opt", np.nan),
                        "bal_accuracy_opt": fold_info.get("bal_accuracy_opt", np.nan),
                        "mcc_opt":          fold_info.get("mcc_opt", np.nan),
                        "pr_gain_opt":      fold_info.get("pr_gain_opt", np.nan),
                        "f1g_opt":          fold_info.get("f1g_opt", np.nan),
                        
                        "best_C":         fold_info.get("best_C", np.nan),
                        "best_penalty":   fold_info.get("best_penalty", ""),
                    })
                    
    df = pd.DataFrame(rows)
    return df.sort_values(
        ["scenario", "representation", "pooling", "fold_id"]
    ).reset_index(drop=True)

### 7. Modelo final

Una vez evaluado el rendimiento por escenario, entrenamos un modelo único con
**todos los datos etiquetados** usando la mejor configuración. Este es el modelo
que se desplegará en inferencia.

Separar evaluación (nested CV) de entrenamiento final es el flujo estándar:
el nested CV estima el rendimiento real; el modelo final maximiza la información
disponible (Cawley & Talbot, 2010, *JMLR*).

In [21]:
def train_final_model(X, y, best_params):
    """
    Entrena un único modelo final sobre TODOS los datos etiquetados.

    :param X:           Array de embeddings completo (n_samples, n_features).
    :param y:           Array de etiquetas completo (n_samples,).
    :param best_params: Dict con hiperparámetros: {'C', 'penalty', 'solver'}.
    :returns: Pipeline entrenado (StandardScaler + LogisticRegression).
    """
    final_model = make_pipeline(
        StandardScaler(),
        LogisticRegression(
            C=best_params["C"],
            penalty=best_params["penalty"],
            solver=best_params["solver"],
            max_iter=5000,
            tol=1e-4,
        )
    )
    final_model.fit(X, y)
    print(f"Modelo final entrenado sobre {len(y)} instancias con params: {best_params}")
    return final_model

### 8. Inferencia sobre datos dudosos y clasificación del nivel de confianza

Tras entrenar el modelo final, se aplica a los pares dudosos (zona Q2-Q3 del ranking
topológico). Para cada pareja dudosa se asigna un **nivel de confianza** de la
predicción según si el modelo ha visto durante el entrenamiento:

- `C1` — mismo grupo efector **y** mismo grupo proteína (máxima confianza).
- `C2E` — mismo grupo efector, grupo proteína nuevo.
- `C2P` — grupo efector nuevo, mismo grupo proteína.
- `C3` — ambos grupos nuevos (mínima confianza; extrapolación total).

Esto sigue directamente el esquema de la Figura 3 del documento de selección de negativos.

In [22]:
def classify_inference_confidence(
    unknown_meta_df: pd.DataFrame,
    train_meta_df: pd.DataFrame,
    eg_col: str = "effector_group",
    pg_col: str = "protein_group",
) -> pd.Series:
    """
    Asigna a cada pareja dudosa su nivel de confianza de predicción (C1/C2E/C2P/C3)
    en función de si el modelo vio durante entrenamiento el grupo de efector y/o proteína.

    :param unknown_meta_df:  DataFrame con los pares dudosos (muestra × metadatos).
    :param train_meta_df:    DataFrame con los pares de entrenamiento.
    :param eg_col:           Nombre de columna de grupo de efector.
    :param pg_col:           Nombre de columna de grupo de proteína.
    :returns: pd.Series con valores 'C1', 'C2E', 'C2P', 'C3' para cada pareja dudosa.
    """
    seen_eg = set(train_meta_df[eg_col])
    seen_pg = set(train_meta_df[pg_col])

    def _level(row):
        has_eg = row[eg_col] in seen_eg
        has_pg = row[pg_col] in seen_pg
        if has_eg and has_pg:
            return "C1"
        elif has_eg and not has_pg:
            return "C2E"
        elif not has_eg and has_pg:
            return "C2P"
        else:
            return "C3"

    return unknown_meta_df.apply(_level, axis=1)


def inference_to_csv(
    final_model,
    X_inference,
    sample_names: list,
    unknown_meta_df: pd.DataFrame,
    train_meta_df: pd.DataFrame,
    output_path,
):
    """
    Genera predicciones sobre muestras dudosas y guarda el resultado en CSV.

    Columnas del CSV:
      sample, effector_group, protein_group, proba_interact,
      predicted_label, confidence_level

    :param final_model:       Pipeline entrenado sobre todos los datos etiquetados.
    :param X_inference:       Array de embeddings de los dudosos (n_samples, n_features).
    :param sample_names:      Lista de nombres de muestra (alineada con X_inference).
    :param unknown_meta_df:   DataFrame con metadatos de los dudosos.
    :param train_meta_df:     DataFrame con metadatos de entrenamiento.
    :param output_path:       Path del CSV de salida.
    :returns: pd.DataFrame con las predicciones ordenadas.
    """
    proba = final_model.predict_proba(X_inference)[:, 1]

    # Alinear metadatos con el orden de X_inference
    meta_aligned = unknown_meta_df.set_index("sample_name").loc[sample_names].reset_index()
    meta_aligned = meta_aligned.rename(columns={"sample_name": "sample"})

    confidence = classify_inference_confidence(meta_aligned, train_meta_df)

    df = meta_aligned.copy()
    df["proba_interact"]  = proba
    df["predicted_label"] = (proba >= 0.5).astype(int)
    df["confidence_level"] = confidence.values

    df = df.sort_values("proba_interact", ascending=False).reset_index(drop=True)

    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_path, index=False)
    print(f"Predicciones guardadas en: {output_path}  ({len(df)} muestras)")
    print("\nDistribución de niveles de confianza:")
    print(df["confidence_level"].value_counts().to_string())
    return df

# Ejecución principal

In [23]:
# ── 0. Cargar metadatos y generar splits Cx ──────────────────────────────────
# metadata.csv debe tener columnas: sample_name, effector, protein,
#                                   effector_group, protein_group, label
#meta_df = pd.read_csv(PATH_METADATA, dtype={"label": int})

# Solo pares con ambas clases en su celda (verdes + naranjas del heatmap P2)
# — los rojos y grises no participan en el entrenamiento ni en los splits
#labeled_meta = meta_df[meta_df["label"].isin([0, 1])].copy()
labeled_meta = df_labeled

# Generar (o regenerar) los splits y guardarlos
splits = build_and_save_splits(
    labeled_meta,
    output_dir=str(SPLITS_DIR),
    effector_group_col="Effector_Group",
    protein_group_col="Protein_Group",
    label_col="Label",
    sample_col="Sample_name",
    min_C3=3,
    min_C2=3,
)

print(f"\nDataset: {len(labeled_meta)} parejas  "
      f"({(labeled_meta.Label==1).sum()} pos, {(labeled_meta.Label==0).sum()} neg)")
print(f"Splits guardados en: {SPLITS_DIR}")


🔧 Generando splits Cx...

📋 Reporte de particiones:

  REPORTE DE PARTICIONES CV — ESCENARIOS Cx
  Umbral C3: ≥3 pos + ≥3 neg por combinación
  Umbral C2: ≥3 pos + ≥3 neg por grupo (celdas biclase)

─────────────────────────────────────────────────────────────────
  Escenario C3  (5 folds válidos)
─────────────────────────────────────────────────────────────────
  Fold [Cluster 1__GO_cluster_08      ]  train= 38  test= 13 (pos=5, neg=8)  excl=146  ✅
  Fold [Cluster 1__GO_cluster_13.1    ]  train= 36  test= 22 (pos=6, neg=16)  excl=139  ✅
  Fold [Cluster 1__GO_cluster_13.3    ]  train= 38  test= 28 (pos=10, neg=18)  excl=131  ✅
  Fold [Cluster 1__GO_cluster_14      ]  train= 38  test= 15 (pos=7, neg=8)  excl=144  ✅
  Fold [Cluster 2__GO_cluster_05      ]  train=114  test= 29 (pos=26, neg=3)  excl= 54  ✅

  → Folds con ambas clases en test: 5/5

─────────────────────────────────────────────────────────────────
  Escenario C2E  (3 folds válidos)
──────────────────────────────────────────

In [24]:
# ── 1. Definir transformaciones de pooling a comparar ────────────────────────
test_transforms = {
    "single_embeddings": [
        lambda x: np.mean(x, axis=0),
        lambda x: np.max(x, axis=0),
        lambda x: np.min(x, axis=0),
    ],
    "pair_embeddings": [
        lambda x: np.mean(x, axis=(0, 1)),
        lambda x: np.max(x, axis=(0, 1)),
        lambda x: np.min(x, axis=(0, 1)),
    ],
}

warnings.filterwarnings("ignore")

In [25]:
# Diagnóstico rápido
nombres_en_disco = [d.name for d in OUTPUT_DIR.iterdir() if d.is_dir() and ".ipynb" not in d.name]
nombres_en_excel = labeled_meta["Sample_name"].unique().tolist()

print(f"Total carpetas en disco: {len(nombres_en_disco)}")
print(f"Total IDs en Excel: {len(nombres_en_excel)}")
print(f"Ejemplo disco: {nombres_en_disco[0] if nombres_en_disco else 'VACÍO'}")
print(f"Ejemplo excel: {nombres_en_excel[0] if nombres_en_excel else 'VACÍO'}")

labeled_meta

Total carpetas en disco: 5472
Total IDs en Excel: 197
Ejemplo disco: P35283_EspL
Ejemplo excel: Q60855_NleB


,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Label,Sample_name
0,NleB,Ripk1,True,Cluster 1,GO_cluster_13.3,1,Q60855_NleB
1,NleB,Nfkb1,True,Cluster 1,GO_cluster_14,1,P25799_NleB
2,NleD,Mapkapk2,True,Cluster 2,GO_cluster_05,1,P49138_NleD
3,NleB,Traf6,True,Cluster 1,GO_cluster_14,1,P70196_NleB
4,NleB,Traf5,True,Cluster 1,GO_cluster_12,1,P70191_NleB
...,...,...,...,...,...,...,...
192,EspS,S100a8,False,Cluster 1,GO_cluster_13.1,0,P27005_EspS
193,EspS,S100a9,False,Cluster 1,GO_cluster_13.1,0,P31725_EspS
194,NleB,Pak5,False,Cluster 1,GO_cluster_05,0,Q8C015_NleB
195,NleE,Pak5,False,Cluster 1,GO_cluster_05,0,Q8C015_NleE


In [26]:
# ── 2. Nested CV para los cuatro escenarios ──────────────────────────────────
# Ejecutar los cuatro escenarios (puede llevar varios minutos según el tamaño del dataset)
all_results = test_pooling_transforms_all_scenarios(
    test_transforms,
    labeled_dir=OUTPUT_DIR,
    splits_dir=SPLITS_DIR,
    metadata_df=labeled_meta,
    scenarios=["C1", "C2E", "C2P", "C3"],
)


######################################################################
#  ESCENARIO C1
######################################################################

  Embedding: single_embeddings  (3 poolings)


✅ Éxito: Se cargaron 197 muestras de las 197 del Excel.
[[[ -296.2   -373.    -234.1  ...   -94.2   -364.8   -336.5 ]
  [ -358.2   -305.8   -165.1  ...   -31.33  -379.2   -384.  ]
  [ -231.5   -417.2   -203.2  ...  -139.    -355.    -290.8 ]
  ...
  [ -274.8   -432.8   -275.5  ...  -118.    -355.2   -369.  ]
  [ -294.8   -437.8   -270.8  ...  -128.4   -347.8   -337.8 ]
  [ -256.8   -409.5   -296.2  ...  -123.1   -429.5   -346.5 ]]

 [[  760.     420.     362.   ...   346.     262.     127.5 ]
  [  510.     504.     378.   ...   520.     192.     189.  ]
  [  506.     352.     352.   ...   322.     402.      93.  ]
  ...
  [  732.     288.     226.   ...   338.     252.     112.  ]
  [  502.     314.     300.   ...   346.     258.     131.  ]
  [  592.     334.     284.   ...   394.     159.     178.  ]]

 [[-1184.   -1004.    -800.   ...  -560.    -996.    -856.  ]
  [-1272.   -1032.    -764.   ...  -524.    -912.    -936.  ]
  [ -956.   -1048.    -808.   ...  -776.    -880.    -904.  

    folds: total=5  válidos=5  NaN=0
    Folds válidos: 5 | Tiempo: 29.1s
    ROC AUC: 0.9093 | PR AUC: 0.8590 | Lift: 2.50x
    AUPRG (Área PR-Gain): 0.8976 | E[FG1] (F1-Gain Esperado): 0.7022
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.7749 | Rec: 0.8527 | Bal.Acc: 0.8568 | PR-Gain: 0.8381 | F1-Gain: 0.8699
      -> Umbral Optimizado [Promedio: 0.550]:
         Prec: 0.8638 | Rec: 0.8659 | Bal.Acc: 0.8940 | PR-Gain: 0.9100 | F1-Gain: 0.9103
    ----------------------------------------------------------

  --- single_embeddings | max ---


    folds: total=5  válidos=5  NaN=0
    Folds válidos: 5 | Tiempo: 1.6s
    ROC AUC: 0.9067 | PR AUC: 0.8581 | Lift: 2.49x
    AUPRG (Área PR-Gain): 0.8723 | E[FG1] (F1-Gain Esperado): 0.6909
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.7571 | Rec: 0.8077 | Bal.Acc: 0.8265 | PR-Gain: 0.8138 | F1-Gain: 0.8426
      -> Umbral Optimizado [Promedio: 0.459]:
         Prec: 0.8240 | Rec: 0.8659 | Bal.Acc: 0.8790 | PR-Gain: 0.8772 | F1-Gain: 0.8939
    ----------------------------------------------------------

  --- single_embeddings | min ---


    folds: total=5  válidos=5  NaN=0
    Folds válidos: 5 | Tiempo: 1.8s
    ROC AUC: 0.9088 | PR AUC: 0.8849 | Lift: 2.57x
    AUPRG (Área PR-Gain): 0.8955 | E[FG1] (F1-Gain Esperado): 0.6978
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.7997 | Rec: 0.8099 | Bal.Acc: 0.8506 | PR-Gain: 0.8659 | F1-Gain: 0.8677
      -> Umbral Optimizado [Promedio: 0.390]:
         Prec: 0.8019 | Rec: 0.8824 | Bal.Acc: 0.8831 | PR-Gain: 0.8686 | F1-Gain: 0.8968
    ----------------------------------------------------------

  Embedding: pair_embeddings  (3 poolings)


...

    folds: total=5  válidos=5  NaN=0
    Folds válidos: 5 | Tiempo: 1.3s
    ROC AUC: 0.4777 | PR AUC: 0.5579 | Lift: 1.25x
    AUPRG (Área PR-Gain): 0.1868 | E[FG1] (F1-Gain Esperado): 0.3475
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.4356 | Rec: 0.5403 | Bal.Acc: 0.4597 | PR-Gain: 0.0633 | F1-Gain: 0.2521
      -> Umbral Optimizado [Promedio: 0.208]:
         Prec: 0.5133 | Rec: 0.8282 | Bal.Acc: 0.5460 | PR-Gain: 0.1815 | F1-Gain: 0.4104
    ----------------------------------------------------------

  Embedding: pair_embeddings  (3 poolings)


✅ Éxito: Se cargaron 197 muestras de las 197 del Excel.
[[[ 5.3516e+00 -1.1152e+00 -7.4414e-01 ...  4.6558e-01 -1.0557e+00
    5.0117e+00]
  [ 6.0234e+00 -4.8125e+00  7.5293e-01 ...  6.7090e-01  5.7324e-01
    5.0430e+00]
  [ 3.7227e+00  4.0938e+00  4.0222e-02 ...  8.1543e-01 -1.4023e+00
    5.2461e+00]
  ...
  [ 4.6172e+00 -1.7256e+00 -1.6875e+00 ...  6.8213e-01 -1.8066e+00
    5.0508e+00]
  [ 3.6367e+00 -1.2939e+00 -1.8330e+00 ...  8.3057e-01 -2.0801e+00
    5.6406e+00]
  [ 3.7930e+00 -1.9006e-01 -1.9756e+00 ...  8.1250e-01 -2.7949e+00
    6.0742e+00]]

 [[ 7.7200e+02  8.8000e+02  7.1200e+02 ...  8.3200e+02  1.0450e+02
    2.1280e+03]
  [ 7.4800e+02  7.9200e+02  7.5200e+02 ...  7.5600e+02  1.0050e+02
    2.0800e+03]
  [ 7.0800e+02  9.1200e+02  7.6800e+02 ...  8.6000e+02  9.5000e+01
    2.2560e+03]
  ...
  [ 7.8400e+02  9.0800e+02  7.4800e+02 ...  8.5200e+02  1.0250e+02
    2.1760e+03]
  [ 7.9200e+02  9.1600e+02  7.7600e+02 ...  8.5600e+02  9.4000e+01
    2.1920e+03]
  [ 8.0800e+02  9

    folds: total=5  válidos=5  NaN=0
    Folds válidos: 5 | Tiempo: 33.8s
    ROC AUC: 0.5297 | PR AUC: 0.5168 | Lift: 1.13x
    AUPRG (Área PR-Gain): 0.1542 | E[FG1] (F1-Gain Esperado): 0.3441
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.5014 | Rec: 0.6615 | Bal.Acc: 0.5217 | PR-Gain: 0.1336 | F1-Gain: 0.2656
      -> Umbral Optimizado [Promedio: 0.319]:
         Prec: 0.5292 | Rec: 0.9569 | Bal.Acc: 0.5854 | PR-Gain: 0.2015 | F1-Gain: 0.4946
    ----------------------------------------------------------

  --- pair_embeddings | max ---


    folds: total=5  válidos=5  NaN=0
    Folds válidos: 5 | Tiempo: 1.1s
    ROC AUC: 0.5694 | PR AUC: 0.6486 | Lift: 1.46x
    AUPRG (Área PR-Gain): 0.2447 | E[FG1] (F1-Gain Esperado): 0.3940
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.6406 | Rec: 0.5481 | Bal.Acc: 0.5768 | PR-Gain: 0.3706 | F1-Gain: 0.3146
      -> Umbral Optimizado [Promedio: 0.283]:
         Prec: 0.6258 | Rec: 0.8600 | Bal.Acc: 0.6460 | PR-Gain: 0.3915 | F1-Gain: 0.6303
    ----------------------------------------------------------

  --- pair_embeddings | min ---


    folds: total=5  válidos=5  NaN=0
    Folds válidos: 5 | Tiempo: 1.1s
    ROC AUC: 0.5221 | PR AUC: 0.6005 | Lift: 1.41x
    AUPRG (Área PR-Gain): 0.2745 | E[FG1] (F1-Gain Esperado): 0.4174
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.4006 | Rec: 0.2977 | Bal.Acc: 0.4815 | PR-Gain: 0.0875 | F1-Gain: 0.0688
      -> Umbral Optimizado [Promedio: 0.299]:
         Prec: 0.6203 | Rec: 0.8248 | Bal.Acc: 0.6547 | PR-Gain: 0.4049 | F1-Gain: 0.6396
    ----------------------------------------------------------


In [27]:
# Guardamos el objeto ante cualquier imprevisto
import joblib

fichero_salida = joblib.dump(all_results, "checkpoint_CV_agrupacion_func_PRG.joblib")

print(f"\n[OK] ¡Análisis de escenarios completado!")
print(f"[OK] Se han guardado todas las métricas, precisiones, recalls y el F1-Gain esperado por fold.")
print(f"[OK] Archivo binario guardado en: '{fichero_salida}'")


[OK] ¡Análisis de escenarios completado!
[OK] Se han guardado todas las métricas, precisiones, recalls y el F1-Gain esperado por fold.
[OK] Archivo binario guardado en: '['checkpoint_CV_agrupacion_func_PRG.joblib']'


In [28]:
# ── 3a. Tabla resumen de resultados por escenario ────────────────────────
df_results = optresults2table_nested(all_results)
display(df_results)
df_results.to_csv(OUTPUT_RESULTS / "cv_results_all_scenarios_g13sep_estricto_PRG.csv", index=False)
print(f"Tabla resumen guardada en: {OUTPUT_RESULTS / 'cv_results_all_scenarios_g13sep_estricto_PRG.csv'}")

,scenario,representation,pooling,roc_auc,pr_auc,pr_auc_lift,auprg,expected_f1_gain,precision_50,recall_50,...,recall_opt,bal_accuracy_opt,mcc_opt,pr_gain_opt,f1_gain_opt,time_s,C_most_freq,penalty,solver,n_valid_folds
0,C1,single_embeddings,min,0.908771,0.884889,2.568233,0.895510,0.697755,0.799744,0.809890,...,0.882418,0.883055,0.751923,0.868580,0.896786,1.783797,1.000,l2,lbfgs,5
1,C1,single_embeddings,mean,0.909314,0.858959,2.496378,0.897572,0.702189,0.774933,0.852747,...,0.865934,0.894044,0.791850,0.910034,0.910321,29.079882,0.100,l2,lbfgs,5
2,C1,single_embeddings,max,0.906651,0.858097,2.487917,0.872280,0.690902,0.757143,0.807692,...,0.865934,0.878967,0.756719,0.877205,0.893906,1.579228,1.000,l1,liblinear,5
3,C1,pair_embeddings,max,0.853374,0.834104,2.420043,0.853115,0.676557,0.716475,0.724176,...,0.781319,0.820967,0.643436,0.822291,0.832106,1.335884,0.010,l2,lbfgs,5
4,C1,pair_embeddings,min,0.837579,0.803883,2.333299,0.807795,0.657096,0.752456,0.719780,...,0.706593,0.818374,0.676172,0.908532,0.833455,1.259220,0.100,l2,lbfgs,5
5,C1,pair_embeddings,mean,0.755256,0.676387,1.961923,0.654693,0.581392,0.548733,0.618681,...,0.764835,0.753341,0.529822,0.702637,0.756073,21.099468,1.000,l2,lbfgs,5
6,C2E,pair_embeddings,min,0.385811,0.586310,1.031155,0.036949,0.269865,0.693276,0.543243,...,0.736937,0.591146,0.204428,0.393716,0.372564,0.582952,0.100,l1,liblinear,3
7,C2E,pair_embeddings,mean,0.500427,0.577698,1.019980,0.265263,0.382632,0.379155,0.576923,...,0.933333,0.550000,0.105409,0.125000,0.510417,17.481208,0.100,l1,liblinear,3
8,C2E,single_embeddings,min,0.347863,0.564472,1.011313,0.240741,0.370370,0.420161,0.489744,...,0.858974,0.429487,-0.088514,0.000000,0.333333,0.679446,0.100,l1,liblinear,3
9,C2E,single_embeddings,max,0.463543,0.553450,0.957561,0.219833,0.360048,0.484036,0.580180,...,0.848649,0.522685,0.051896,0.068280,0.392363,0.654119,0.001,l2,lbfgs,3


Tabla resumen guardada en: results_NestedCV_Cx/results_NestedCV_nivel_estricto_g13sep/cv_results_all_scenarios_g13sep_estricto_PRG.csv


In [29]:
# ── 3b. Tabla detallada por fold individual ───────────────────────────────
# Muestra para cada fold: grupo en test, composición (n_pos/n_neg) y métricas.
# Especialmente útil en C3 con pocos folds: la varianza entre folds es
# información real sobre robustez que hay que reportar (Chicco & Jurman, 2020).
df_folds = fold_detail_table(all_results)
#df_folds = pd.read_csv(OUTPUT_RESULTS / "cv_results_per_fold_g13sep_estricto.csv")

# Imprimir la mejor combinación embedding×pooling por escenario
import numpy as np

# Definimos las columnas que queremos inspeccionar al detalle por cada fold
# Incluimos la comparativa de umbrales (50 vs opt), precisión, recall, lift, auprg y e[fg1]
cols = [
    "fold_id", "note", "n_test", "n_test_pos",
    "roc_auc", "pr_auc", "lift", "auprg", "expected_f1g",
    "best_threshold", "precision_50", "recall_50", "f1g_50",
    "precision_opt", "recall_opt", "f1g_opt", "best_C"
]

for scenario in ["C3", "C2E", "C2P", "C1"]:
    if scenario not in df_results["scenario"].values:
        continue
        
    # === MODIFICACIÓN 1: Ordenamos por 'auprg' (métricamente más robusto que pr_auc) ===
    best_combo = (
        df_results[df_results["scenario"] == scenario]
        .sort_values("auprg", ascending=False)
        .iloc[0]
    )

    # Filtrar el detalle de folds para el mejor modelo de este escenario
    sub = df_folds[
        (df_folds["scenario"]       == scenario) &
        (df_folds["representation"] == best_combo["representation"]) &
        (df_folds["pooling"]        == best_combo["pooling"])
    ]

    # === MODIFICACIÓN 2: Calcular el Error Estándar (SEM) sobre el AUPRG ===
    n_folds = len(sub)
    # Usamos ddof=1 para la desviación estándar muestral unbiased
    auprg_std = sub['auprg'].std(ddof=1) / np.sqrt(n_folds) if n_folds > 1 else 0.0

    print(f"\n{'='*85}")
    print(f"  {scenario} — {best_combo['representation']} | {best_combo['pooling']}")
    print(f"  (Media AUPRG = {best_combo['auprg']:.4f} ± {auprg_std:.4f} | E[FG1] = {best_combo['expected_f1_gain']:.4f})")
    print(f"{'='*85}")
    
    # Asegurar que solo imprimimos las columnas que realmente existen en el DataFrame
    existing_cols = [c for c in cols if c in sub.columns]
    display(sub[existing_cols].reset_index(drop=True))

# Guardar el archivo final en el directorio de salida correspondiente
output_path = OUTPUT_RESULTS / "cv_results_per_fold_g13sep_estricto_PRG.csv"
df_folds.to_csv(output_path, index=False)
print(f"\nTabla completa por fold guardada en: {output_path}")


  C3 — single_embeddings | mean
  (Media AUPRG = 0.3021 ± 0.0786 | E[FG1] = 0.4088)


,fold_id,note,n_test,n_test_pos,roc_auc,pr_auc,lift,auprg,expected_f1g,best_threshold,precision_50,recall_50,f1g_50,precision_opt,recall_opt,f1g_opt,best_C
0,Cluster 1__GO_cluster_08,ok,13,5,0.6500,0.6909,1.7964,0.5193,0.5097,0.2377,0.5000,0.4000,0.2187,0.6000,0.6000,0.5833,10.00
1,Cluster 1__GO_cluster_13.1,ok,22,6,0.5729,0.3187,1.1684,0.1105,0.3259,0.2575,0.3571,0.8333,0.6250,0.3750,1.0000,0.6875,10.00
2,Cluster 1__GO_cluster_13.3,ok,28,10,0.6389,0.5646,1.5809,0.3280,0.4271,0.0397,0.5714,0.4000,0.3750,0.4615,0.6000,0.4907,10.00
3,Cluster 1__GO_cluster_14,ok,15,7,0.6786,0.6345,1.3597,0.4139,0.4617,0.7524,0.5455,0.8571,0.5625,0.6000,0.8571,0.6354,10.00
4,Cluster 2__GO_cluster_05,ok,29,26,0.5897,0.9423,1.0510,0.1389,0.3194,0.0100,0.8889,0.9231,0.0972,0.8966,1.0000,0.5000,0.01



  C2E — pair_embeddings | max
  (Media AUPRG = 0.3364 ± 0.1636 | E[FG1] = 0.4187)


,fold_id,note,n_test,n_test_pos,roc_auc,pr_auc,lift,auprg,expected_f1g,best_threshold,precision_50,recall_50,f1g_50,precision_opt,recall_opt,f1g_opt,best_C
0,Cluster 1,ok,159,37,0.3934,0.1857,0.7981,0.0091,0.256,0.4357,0.1795,0.3784,0.0577,0.2500,0.7838,0.5033,0.001
1,Cluster 2,ok,29,26,0.5000,0.8966,1.0000,0.5000,0.500,0.0100,0.8966,1.0000,0.5000,0.8966,1.0000,0.5000,0.001
2,Cluster 6,ok,9,5,0.5000,0.5556,1.0000,0.5000,0.500,0.0100,0.5556,1.0000,0.5000,0.5556,1.0000,0.5000,0.001



  C2P — single_embeddings | min
  (Media AUPRG = 0.7990 ± 0.0756 | E[FG1] = 0.6573)


,fold_id,note,n_test,n_test_pos,roc_auc,pr_auc,lift,auprg,expected_f1g,best_threshold,precision_50,recall_50,f1g_50,precision_opt,recall_opt,f1g_opt,best_C
0,GO_cluster_05,ok,83,30,0.7352,0.5597,1.5485,0.5200,0.5490,0.3070,0.5429,0.6333,0.5978,0.5349,0.7667,0.6678,1.0
1,GO_cluster_08,ok,13,5,0.8250,0.8833,2.2967,0.7729,0.6365,0.2575,1.0000,0.6000,0.7917,1.0000,0.8000,0.9219,0.1
2,GO_cluster_13.1,ok,24,7,0.9076,0.8028,2.7525,0.8442,0.6721,0.0793,0.5556,0.7143,0.7529,0.6364,1.0000,0.8824,10.0
3,GO_cluster_13.3,ok,28,10,0.9611,0.9351,2.6184,0.9309,0.7154,0.3070,0.7500,0.9000,0.8765,0.7692,1.0000,0.9167,1.0
4,GO_cluster_14,ok,15,7,1.0000,1.0000,2.1429,0.9271,0.7135,0.0397,1.0000,0.7143,0.8250,1.0000,1.0000,1.0000,10.0



  C1 — single_embeddings | mean
  (Media AUPRG = 0.8976 ± 0.0332 | E[FG1] = 0.7022)


,fold_id,note,n_test,n_test_pos,roc_auc,pr_auc,lift,auprg,expected_f1g,best_threshold,precision_50,recall_50,f1g_50,precision_opt,recall_opt,f1g_opt,best_C
0,fold_0,ok,40,14,0.9093,0.9255,2.6443,0.9472,0.7236,0.3367,0.9167,0.7857,0.9021,0.9231,0.8571,0.9327,0.1
1,fold_1,ok,40,14,0.9451,0.8892,2.5405,0.8999,0.7000,0.5148,0.7368,1.0000,0.9038,0.7778,1.0000,0.9231,0.1
2,fold_2,ok,39,14,0.8371,0.6685,1.8622,0.7772,0.6556,0.3961,0.8462,0.7857,0.8727,0.8000,0.8571,0.8833,1.0
3,fold_3,ok,39,13,0.9615,0.9615,2.8846,0.9687,0.7343,0.6435,0.7500,0.9231,0.8958,1.0000,0.9231,0.9792,0.1
4,fold_4,ok,39,13,0.8935,0.8501,2.5502,0.8949,0.6974,0.8613,0.6250,0.7692,0.7750,0.8182,0.6923,0.8333,1.0



Tabla completa por fold guardada en: results_NestedCV_Cx/results_NestedCV_nivel_estricto_g13sep/cv_results_per_fold_g13sep_estricto_PRG.csv


In [30]:
# Volvemos a cargar los resultados para no tener que ejecutar de nuevo todo el análisis
#df_results = pd.read_csv(OUTPUT_RESULTS / "cv_results_all_scenarios_g13sep_estricto.csv")

In [31]:
# ── 4. Elegir mejor configuración (por PR AUC en C3 — escenario más exigente) ─
# Puede cambiarse a "C1" para el baseline o a otro escenario según criterio biológico
TARGET_SCENARIO = "C3"

# === MODIFICACIÓN 1: Ordenamos por 'auprg' (Área de PR-Gain) ===
best_row = (
    df_results[df_results["scenario"] == TARGET_SCENARIO]
    .sort_values("auprg", ascending=False)
    .iloc[0]
)

BEST_EMB_TYPE    = best_row["representation"]
BEST_POOLING_IDX = ["mean", "max", "min"].index(best_row["pooling"])
BEST_TRANSFORM   = test_transforms[BEST_EMB_TYPE][BEST_POOLING_IDX]

best_params = {
    "C":       best_row["C_most_freq"],
    "penalty": best_row["penalty"],
    "solver":  best_row["solver"],
}

print(f"========================================================================")
print(f" MEJOR CONFIGURACIÓN ENCONTRADA ({TARGET_SCENARIO})")
print(f"========================================================================")
print(f"  Embedding:  {BEST_EMB_TYPE}")
print(f"  Pooling:    {best_row['pooling']}")
print(f"  Hiperparam: {best_params}")
print(f"------------------------------------------------------------------------")
print(f"  [MÉTRICAS GLOBALES DE LA CURVA]")
print(f"    AUPRG (PR-Gain AUC):     {best_row['auprg']:.4f}  (Métrica de selección)")
print(f"    E[FG1] (F1-Gain Esperado):{best_row['expected_f1_gain']:.4f}")
print(f"    PR AUC (Absoluto):       {best_row['pr_auc']:.4f}  |  PR AUC Lift: {best_row['pr_auc_lift']:.2f}x")
print(f"    ROC AUC:                 {best_row['roc_auc']:.4f}")
print(f"------------------------------------------------------------------------")
print(f"  [RENDIMIENTO EN PUNTOS OPERATIVOS SELECCIONADOS]")
print(f"    -> Con Umbral Fijo [0.50]:")
print(f"       Bal. Accuracy: {best_row['bal_accuracy_50']:.4f}  |  MCC: {best_row['mcc_50']:.4f}")
print(f"       PR-Gain Pct:   {best_row['pr_gain_50']:.4f}  |  F1-Gain: {best_row['f1_gain_50']:.4f}")
print(f"       Precision:     {best_row['precision_50']:.4f}  |  Recall: {best_row['recall_50']:.4f}")
print(f"    -> Con Umbral Optimizado [Umbral Promedio: {best_row['best_threshold']:.3f}]:")
print(f"       Bal. Accuracy: {best_row['bal_accuracy_opt']:.4f}  |  MCC: {best_row['mcc_opt']:.4f}")
print(f"       PR-Gain Pct:   {best_row['pr_gain_opt']:.4f}  |  F1-Gain: {best_row['f1_gain_opt']:.4f}")
print(f"       Precision:     {best_row['precision_opt']:.4f}  |  Recall: {best_row['recall_opt']:.4f}")
print(f"========================================================================")

 MEJOR CONFIGURACIÓN ENCONTRADA (C3)
  Embedding:  single_embeddings
  Pooling:    mean
  Hiperparam: {'C': 10.0, 'penalty': 'l2', 'solver': 'newton-cg'}
------------------------------------------------------------------------
  [MÉTRICAS GLOBALES DE LA CURVA]
    AUPRG (PR-Gain AUC):     0.3021  (Métrica de selección)
    E[FG1] (F1-Gain Esperado):0.4088
    PR AUC (Absoluto):       0.6302  |  PR AUC Lift: 1.39x
    ROC AUC:                 0.6260
------------------------------------------------------------------------
  [RENDIMIENTO EN PUNTOS OPERATIVOS SELECCIONADOS]
    -> Con Umbral Fijo [0.50]:
       Bal. Accuracy: 0.5809  |  MCC: 0.1673
       PR-Gain Pct:   0.3108  |  F1-Gain: 0.3757
       Precision:     0.5726  |  Recall: 0.6827
    -> Con Umbral Optimizado [Umbral Promedio: 0.259]:
       Bal. Accuracy: 0.6293  |  MCC: 0.2612
       PR-Gain Pct:   0.3454  |  F1-Gain: 0.5794
       Precision:     0.5866  |  Recall: 0.8114


In [32]:
# # ── 5. Sanity check con la mejor representación ──────────────────────────────
# X_labeled, y_labeled, names_labeled = load_block_from_csv(
#     OUTPUT_DIR,
#     df_labeled,
#     emb_type=BEST_EMB_TYPE,
#     transforms=[BEST_TRANSFORM],
# )
# Xi = X_labeled[:, 0, :] if X_labeled.ndim == 3 else X_labeled

# sanity_check_random_labels(Xi, y_labeled)

In [33]:
# ── 6. Modelo final: entrenado con TODOS los datos etiquetados ───────────────
# final_model = train_final_model(Xi, y_labeled, best_params)

In [34]:
# # ── 7. Inferencia sobre datos dudosos ────────────────────────────────────────
# # Cargar embeddings de los pares dudosos (zona Q2-Q3 del ranking topológico)
# X_unknown, _, names_unknown = load_block_from_csv(
#     OUTPUT_DIR,
#     df_unknown,
#     emb_type=BEST_EMB_TYPE,
#     transforms=[BEST_TRANSFORM],
# )
# Xi_unknown = X_unknown[:, 0, :] if X_unknown.ndim == 3 else X_unknown

# # Metadatos de los dudosos (debe existir: sample_name, effector_group, protein_group)
# unknown_meta = df_total[
#     lambda d: d["Sample_name"].isin(names_unknown)
# ].copy()

# # Metadatos de entrenamiento para clasificar nivel de confianza
# train_meta = labeled_meta.copy()

# df_inference = inference_to_csv(
#     final_model,
#     Xi_unknown,
#     names_unknown,
#     unknown_meta_df=unknown_meta,
#     train_meta_df=train_meta,
#     output_path=OUTPUT_DIR / "predictions_unknown.csv",
# )
# display(df_inference.head(20))

In [35]:
# # ── 8. Resumen final por nivel de confianza ──────────────────────────────────
# summary = (
#     df_inference
#     .groupby("confidence_level")
#     .agg(
#         n_pares=("sample", "count"),
#         n_predicted_pos=("predicted_label", "sum"),
#         proba_mean=("proba_interact", "mean"),
#         proba_std=("proba_interact", "std"),
#     )
#     .reindex(["C1", "C2E", "C2P", "C3"])  # orden de mayor a menor confianza
#     .reset_index()
# )
# print("\nResumen de inferencia por nivel de confianza:")
# display(summary)